In [ ]:
# ============================================================
# BUILD FACT_SESSION_RESULTS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

# p_batch_id = "20250101_090000"

def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/04_gold_helpers.ipynb")

In [ ]:
# -----------------------------------------
# p_batch_id
# -----------------------------------------
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")
else:
    raise Exception("❌ p_batch_id not provided")

print("Using p_batch_id:", p_batch_id)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:09:23 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:09:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:09:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:09:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:09:24 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:09:24 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:09:24 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [1]:
# -----------------------------------------
# Paths
# -----------------------------------------
silver_results = f"{SILVER_PATH}/results/data.parquet"
silver_sprints = f"{SILVER_PATH}/sprints/data.parquet"

gold_path = f"{GOLD_PATH}/fact_session_results"
os.makedirs(gold_path, exist_ok=True)

NameError: name 'SILVER_PATH' is not defined

In [ ]:
# -----------------------------------------
# Read
# -----------------------------------------

results_df = (
    spark.read.parquet(silver_results)
        .filter(F.col("batch_id") == p_batch_id)
        .withColumn("session_type", F.lit("RACE"))
)

sprints_df = (
    spark.read.parquet(silver_sprints)
        .filter(F.col("batch_id") == p_batch_id)
        .withColumn("session_type", F.lit("SPRINT"))
)

if results_df.count() == 0 and sprints_df.count() == 0:
    raise Exception("❌ Neither results nor sprints have rows for this batch_id")

In [ ]:
# Union
combined_df = results_df.unionByName(sprints_df)

In [ ]:
# Derived columns
fact_df = (
    combined_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

✔ fact_session_results written to: /Users/manoharazalki/F1-ANALYTICS/data/gold/fact_session_results/fact_session_results.parquet
+------+-----+--------------+-----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+------------+--------------------------+-------------------------------------------------------------+------------+------+---------+----------+
|season|round|constructor_id|driver_id  |race_date |race_name         |grid_position|completed_laps|car_number|points|final_position|final_position_text|status      |ingest_timestamp          |source_file                                                  |session_type|is_win|is_podium|has_points|
+------+-----+--------------+-----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+------------+--------------------------+-------------------------------------------------------------+------------+------+--

In [ ]:
# Write
write_to_gold(
    input_df=fact_df,
    target_path=f"{gold_path}/data.parquet",
    merge_keys=["season", "round", "constructor_id", "driver_id", "session_type"]
)

print("✔ fact_session_results written")

spark.read.parquet(f"{gold_path}/data.parquet").show(5, truncate=False)